# Aggregated News Comparative Study

AG is made up of more than 1 million news articles, gathered from more than 2000 news sources by academic news search engine (ComeToMyHead) over more than a year.

Each example is pair of (news article, label)
Labels:
| ID | Label |
| -------- | -------- |
|0| World|
|1| Sports|
|2| Business|
|3| Sci/Tech|


**Authors:** Anh Nguyen, Berke Kara

Acknowledgement:
We appreciate fancyzhx for creating and sharing the AG News dataset for research.

#Pre-Process

In [33]:
!pip install gensim datasets

from collections import Counter
from datasets import load_dataset,concatenate_datasets,Dataset, DatasetDict
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
from gensim.utils import simple_preprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import pipeline
from transformers_interpret import SequenceClassificationExplainer


In [34]:
ds = load_dataset("fancyzhx/ag_news")
ds_combined = concatenate_datasets([ds["train"], ds["test"]])
df = pd.DataFrame(ds_combined)
w2v_model = api.load("word2vec-google-news-300")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline_device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {device}")

# Global Label Mappings
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {v: k for k, v in id2label.items()}

Using device: cuda


In [35]:
df_sampled, _ = train_test_split(
    df,
    train_size=30000,
    random_state=42,
    stratify=df["label"]
)

train_df, temp_df = train_test_split(
    df_sampled,
    test_size=0.3,
    random_state=42,
    stratify=df_sampled["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

In [36]:
X_train = train_df['text']
y_train = train_df['label']

X_val = val_df['text']
y_val = val_df['label']

X_test = test_df['text']
y_test = test_df['label']

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_val)}")
print(f"Test examples: {len(X_test)}")

Training examples: 21000
Validation examples: 4500
Test examples: 4500


# Model 1: Classical Baseline

In [37]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

val_preds = model.predict(X_val_vec)
test_preds = model.predict(X_test_vec)

print("Model 1: Classical Baseline Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(y_val, val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Test Macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=list(id2label.values()), digits=4))

Model 1: Classical Baseline Results
--------------------------------------------------
Validation Accuracy: 0.8931
Validation Macro F1: 0.8927
Test Accuracy: 0.9036
Test Macro F1: 0.9032

Classification Report:
              precision    recall  f1-score   support

       World     0.9181    0.8764    0.8968      1125
      Sports     0.9363    0.9804    0.9579      1125
    Business     0.8883    0.8693    0.8787      1125
    Sci/Tech     0.8710    0.8880    0.8794      1125

    accuracy                         0.9036      4500
   macro avg     0.9034    0.9036    0.9032      4500
weighted avg     0.9034    0.9036    0.9032      4500



# Model 2: Pretrained Word2Vec with Classifier

In [38]:
def text_to_vector(text, model, dim=300):
    """
    Convert a sentence/document to a fixed-size vector
    by averaging all word vectors (mean pooling).
    """
    tokens = simple_preprocess(text)

    valid_vectors = [model[token] for token in tokens if token in model]

    if not valid_vectors:
        return np.zeros(dim)

    return np.mean(valid_vectors, axis=0)


In [39]:
X = np.array([text_to_vector(text, w2v_model) for text in X_train])
y = np.array(y_train)

clf = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=1000)

clf.fit(X, y)
print("Classifier trained")


Classifier trained


In [40]:
X_test_vec = np.array([text_to_vector(text, w2v_model) for text in X_test])
X_val_vec = np.array([text_to_vector(text, w2v_model) for text in X_val])

val_preds = clf.predict(X_val_vec)
test_preds = clf.predict(X_test_vec)

print("Model 2: Pretrained Word2Vec + Classifier Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(y_val, val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Test Macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=list(id2label.values()), digits=4))

Model 2: Pretrained Word2Vec + Classifier Results
--------------------------------------------------
Validation Accuracy: 0.8793
Validation Macro F1: 0.8792
Test Accuracy: 0.8853
Test Macro F1: 0.8852

Classification Report:
              precision    recall  f1-score   support

       World     0.8828    0.8702    0.8765      1125
      Sports     0.9509    0.9636    0.9572      1125
    Business     0.8612    0.8382    0.8495      1125
    Sci/Tech     0.8460    0.8693    0.8575      1125

    accuracy                         0.8853      4500
   macro avg     0.8852    0.8853    0.8852      4500
weighted avg     0.8852    0.8853    0.8852      4500



# Model 3: LSTM from Scratch

In [41]:
from torch.nn.utils.rnn import pad_sequence

# 1. Build Vocabulary
def build_vocab(texts, max_size=20000):
    counter = Counter()
    for text in texts:
        counter.update(simple_preprocess(text))

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(X_train)
print(f"Vocabulary size: {len(vocab)}")

# 2. Dataset Class
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = simple_preprocess(self.texts[idx])
        tokens = tokens[:self.max_length]
        indices = [self.vocab.get(token, self.vocab['<UNK>']) for token in tokens]
        if len(indices) == 0:
            indices = [self.vocab['<UNK>']] # Handle empty sequences
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

# 3. Collate Function (for padding)
def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    # Pad sequences to match the longest one in the batch
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded_sequences, lengths, torch.stack(labels)

# 4. Create DataLoaders
train_dataset = TextDataset(X_train, y_train, vocab)
val_dataset = TextDataset(X_val, y_val, vocab)
test_dataset = TextDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"Created train_loader with {len(train_loader)} batches.")
print(f"Created val_loader with {len(val_loader)} batches.")
print(f"Created test_loader with {len(test_loader)} batches.")

Vocabulary size: 20000
Created train_loader with 657 batches.
Created val_loader with 141 batches.
Created test_loader with 141 batches.


In [42]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))
        # embedded: (batch, seq_len, embed_dim)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, (hidden, _) = self.lstm(packed)

        # hidden: (num_layers * 2, batch, hidden_dim)
        forward_hidden  = hidden[-2]   # last layer, forward
        backward_hidden = hidden[-1]   # last layer, backward

        # Concatenate both directions
        combined = torch.cat([forward_hidden, backward_hidden], dim=1)
        # combined: (batch, hidden_dim * 2)

        out = self.dropout(combined)
        # (batch, num_classes)
        return self.fc(out)

model = LSTMClassifier(
    vocab_size  = len(vocab),
    embed_dim   = 128,
    hidden_dim  = 256,
    num_classes = 4,
    num_layers  = 2,
    dropout     = 0.3
).to(device)

# Count parameters — report this in your paper
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


Trainable parameters: 4,929,540


In [43]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for sequences, lengths, labels in loader:
        sequences, lengths, labels = (
            sequences.to(device),
            lengths.to(device),
            labels.to(device)
        )

        optimizer.zero_grad()
        outputs = model(sequences, lengths)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping — important for LSTM stability
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for sequences, lengths, labels in loader:
            sequences, lengths, labels = (
                sequences.to(device),
                lengths.to(device),
                labels.to(device)
            )
            outputs = model(sequences, lengths)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

# Train
EPOCHS = 10
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion) # Changed to val_loader
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

Epoch 01 | Train Loss: 0.8360 Acc: 0.6615 | Val Loss: 0.5229 Acc: 0.8149
Epoch 02 | Train Loss: 0.4706 Acc: 0.8344 | Val Loss: 0.4719 Acc: 0.8511
Epoch 03 | Train Loss: 0.3448 Acc: 0.8824 | Val Loss: 0.4705 Acc: 0.8518
Epoch 04 | Train Loss: 0.2703 Acc: 0.9074 | Val Loss: 0.4037 Acc: 0.8773
Epoch 05 | Train Loss: 0.2091 Acc: 0.9273 | Val Loss: 0.4014 Acc: 0.8820
Epoch 06 | Train Loss: 0.1693 Acc: 0.9420 | Val Loss: 0.4337 Acc: 0.8793
Epoch 07 | Train Loss: 0.1297 Acc: 0.9563 | Val Loss: 0.4564 Acc: 0.8847
Epoch 08 | Train Loss: 0.1067 Acc: 0.9622 | Val Loss: 0.4803 Acc: 0.8902
Epoch 09 | Train Loss: 0.0589 Acc: 0.9791 | Val Loss: 0.5002 Acc: 0.8909
Epoch 10 | Train Loss: 0.0495 Acc: 0.9820 | Val Loss: 0.5274 Acc: 0.8909


In [44]:
def get_preds(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sequences, lengths, labels in loader:
            outputs = model(sequences.to(device), lengths.to(device))
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return all_labels, all_preds

val_labels, val_preds = get_preds(model, val_loader)
test_labels, test_preds = get_preds(model, test_loader)

print("Model 3: LSTM from Scratch Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(val_labels, val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(val_labels, val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print(f"Test Macro F1: {f1_score(test_labels, test_preds, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds,
      target_names=list(id2label.values()), digits=4))

Model 3: LSTM from Scratch Results
--------------------------------------------------
Validation Accuracy: 0.8909
Validation Macro F1: 0.8906
Test Accuracy: 0.8924
Test Macro F1: 0.8921

Classification Report:
              precision    recall  f1-score   support

       World     0.9240    0.8640    0.8930      1125
      Sports     0.9291    0.9787    0.9532      1125
    Business     0.8489    0.8738    0.8611      1125
    Sci/Tech     0.8688    0.8533    0.8610      1125

    accuracy                         0.8924      4500
   macro avg     0.8927    0.8924    0.8921      4500
weighted avg     0.8927    0.8924    0.8921      4500



# Model 4: Pretrained Model with Fine-Tuning
---



In [45]:
import datasets # Import module to avoid conflict with torch.utils.data.Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

dataset = datasets.DatasetDict({
    "train": datasets.Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": datasets.Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False),
    "test": datasets.Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./distilbert_agnews",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

trainer.train()

# val results
predictions = trainer.predict(tokenized_dataset["validation"])
val_logits = predictions.predictions
val_labels = predictions.label_ids
val_preds = np.argmax(val_logits, axis=1)

# test results
test_preds_output = trainer.predict(tokenized_dataset["test"])
test_logits = test_preds_output.predictions
test_labels = test_preds_output.label_ids
test_preds = np.argmax(test_logits, axis=1)

print("Model 4: Pretrained Model with Fine-Tuning Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(val_labels, val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(val_labels, val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print(f"Test Macro F1: {f1_score(test_labels, test_preds, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=list(id2label.values()), digits=4))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/21000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model 4: Pretrained Model with Fine-Tuning Results
--------------------------------------------------
Validation Accuracy: 0.9258
Validation Macro F1: 0.9257
Test Accuracy: 0.9296
Test Macro F1: 0.9296

Classification Report:
              precision    recall  f1-score   support

       World     0.9345    0.9262    0.9304      1125
      Sports     0.9857    0.9822    0.9840      1125
    Business     0.9066    0.8969    0.9017      1125
    Sci/Tech     0.8923    0.9129    0.9025      1125

    accuracy                         0.9296      4500
   macro avg     0.9298    0.9296    0.9296      4500
weighted avg     0.9298    0.9296    0.9296      4500



In [56]:
!pip install transformers-interpret captum

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 110.1 MB/s eta 0:00:00


### Interpretability with Integrated Gradients

Integrated Gradients for our DistilBERT model. Attribution score for each word, positive scores push toward the predicted class, negative scores push away.

In [57]:
model.config.id2label = id2label
model.config.label2id = label2id

cls_explainer = SequenceClassificationExplainer(
    model,
    tokenizer
)

# Take the first example from the test set
sample_text = test_df["text"].iloc[0]
true_label = test_df["label"].iloc[0]

print(f"Original Text: {sample_text}")
print(f"True Category: {id2label[true_label]}\n")

# Calculate attributions
word_attributions = cls_explainer(sample_text)

# Visualize the attributions in an HTML format directly in Colab
cls_explainer.visualize()

Original Text: Owen Signals Changing Champions League Fortunes  LONDON (Reuters) - Michael Owen scored his first goal for  Real Madrid on a Tuesday night of changing fortunes right  across the Champions League.
True Category: Sports



# Model 5: pretrained model without task-specific training

In [59]:
candidate_labels = list(label2id.keys())

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=pipeline_device
)

def predict_zero_shot(texts, batch_size=16):
    preds = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        outputs = zero_shot_classifier(
            batch,
            candidate_labels=candidate_labels,
            hypothesis_template="This news article is about {}.",
            multi_label=False
        )

        if isinstance(outputs, dict):
            outputs = [outputs]

        for output in outputs:
            pred_label = output["labels"][0]
            preds.append(label2id[pred_label])

        print(f"Processed {min(i + batch_size, len(texts))}/{len(texts)} examples")

    return np.array(preds)

Using device: GPU


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [47]:
val_texts = X_val.tolist()
val_labels = y_val.to_numpy()

zero_val_preds = predict_zero_shot(val_texts, batch_size=32)

Processed 32/4500 examples
Processed 64/4500 examples
Processed 96/4500 examples
Processed 128/4500 examples
Processed 160/4500 examples
Processed 192/4500 examples
Processed 224/4500 examples
Processed 256/4500 examples
Processed 288/4500 examples
Processed 320/4500 examples
Processed 352/4500 examples
Processed 384/4500 examples
Processed 416/4500 examples
Processed 448/4500 examples
Processed 480/4500 examples
Processed 512/4500 examples
Processed 544/4500 examples
Processed 576/4500 examples
Processed 608/4500 examples
Processed 640/4500 examples
Processed 672/4500 examples
Processed 704/4500 examples
Processed 736/4500 examples
Processed 768/4500 examples
Processed 800/4500 examples
Processed 832/4500 examples
Processed 864/4500 examples
Processed 896/4500 examples
Processed 928/4500 examples
Processed 960/4500 examples
Processed 992/4500 examples
Processed 1024/4500 examples
Processed 1056/4500 examples
Processed 1088/4500 examples
Processed 1120/4500 examples
Processed 1152/4500

In [48]:
test_texts = X_test.tolist()
test_labels = y_test.to_numpy()

zero_test_preds = predict_zero_shot(test_texts, batch_size=16)

print("Model 5: Zero-Shot Classification Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(val_labels, zero_val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(val_labels, zero_val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(test_labels, zero_test_preds):.4f}")
print(f"Test Macro F1: {f1_score(test_labels, zero_test_preds, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(
    test_labels,
    zero_test_preds,
    target_names=list(id2label.values()),
    digits=4
))

Processed 16/4500 examples
Processed 32/4500 examples
Processed 48/4500 examples
Processed 64/4500 examples
Processed 80/4500 examples
Processed 96/4500 examples
Processed 112/4500 examples
Processed 128/4500 examples
Processed 144/4500 examples
Processed 160/4500 examples
Processed 176/4500 examples
Processed 192/4500 examples
Processed 208/4500 examples
Processed 224/4500 examples
Processed 240/4500 examples
Processed 256/4500 examples
Processed 272/4500 examples
Processed 288/4500 examples
Processed 304/4500 examples
Processed 320/4500 examples
Processed 336/4500 examples
Processed 352/4500 examples
Processed 368/4500 examples
Processed 384/4500 examples
Processed 400/4500 examples
Processed 416/4500 examples
Processed 432/4500 examples
Processed 448/4500 examples
Processed 464/4500 examples
Processed 480/4500 examples
Processed 496/4500 examples
Processed 512/4500 examples
Processed 528/4500 examples
Processed 544/4500 examples
Processed 560/4500 examples
Processed 576/4500 example

### Error Analysis (Best Model: DistilBERT)


In [61]:
# test_preds currently holds the predictions from Model 4 (DistilBERT)
# test_labels holds the true labels
incorrect_indices = np.where(test_preds != test_labels)[0]

# Create a DataFrame to easily view the errors using global id2label
error_data = {
    "Text": [test_df["text"].iloc[i] for i in incorrect_indices],
    "True_Label": [id2label[test_labels[i]] for i in incorrect_indices],
    "Predicted_Label": [id2label[test_preds[i]] for i in incorrect_indices]
}

error_df = pd.DataFrame(error_data)

print(f"Total misclassifications by DistilBERT: {len(error_df)} out of {len(test_labels)}")
print("Displaying the first 20 errors:\n")
print("-"*50)
# Set pandas display options so text isn't truncated
pd.set_option('display.max_colwidth', None)
display(error_df.head(20))

# Reset display options afterward
pd.reset_option('display.max_colwidth')

Total misclassifications by DistilBERT: 317 out of 4500
Displaying the first 20 errors:

--------------------------------------------------


,Text,True_Label,Predicted_Label
0,"US economy generated 144,000 jobs in August (AFP) AFP - The US economy generated 144,000 jobs in August, the Labour Department said, in a sign that the labour market was improving slightly after two sluggish months.",Business,World
1,"Lessons Airlines Can Learn From PCs Without radical strategic change, the legacy carriers won't survive.",Business,Sci/Tech
2,"Kodak Seeks \$1 Billion from Sun in Java Suit Eastman Kodak #39;s patent-infringement lawsuit against Sun Microsystems now moves into the penalty phase, as the jury considers whether to award Kodak as much as \$1 billion in damages.",Sci/Tech,Business
3,"Amazon launching DVD rentals The UK website of Amazon.com is launching a DVD rental service, the online retailer #39;s first foray into a market pioneered by US- based Netflix.",Business,Sci/Tech
4,"Ethics code written to reprogram tech industry Hewlett-Packard, IBM and Dell joined a host of electronics makers Thursday in an effort to promote a unified code of socially responsible business practices across the world.",Business,Sci/Tech
5,"Spam e-mails tempt net shoppers \Computer users continue to ignore warnings about spam and are being lured into buying goods, a report suggests.",World,Sci/Tech
6,Finance Leaders Tackle Terror Financing (Reuters) Reuters - World finance ministers gathered\under heavy guard on Sunday to discuss efforts to fight terror\financing while warning the world must stay focused on the\economic recovery and fight against poverty.,Business,World
7,"Peoria nurse coordinator gets honor Cindy Staggs, the Peoria Unified School District #39;s nurse coordinator and a nurse at Raymond S. Kellis High School, has been named School Nurse Administrator of the Year by the School Nurse Organization of Arizona.",Sci/Tech,Business
8,"US Begins Criminal Probe on Riggs-Paper WASHINGTON (Reuters) - The US Justice Department has begun a criminal investigation into possible wrongdoing at Riggs Bank, The Washington Post reported on Saturday, citing a letter from the US attorney for the District of Columbia to federal bank ...",Business,World
9,"No winners in Powerball, Hoosier Lotto INDIANAPOLIS -- None of the tickets sold for the latest Hoosier Lotto game matched all six numbers drawn, so the jackpot grows to an estimated \$7 million for Wednesday #39;s drawing.",Business,Sports


### Error Analysis


**1. Blurring between Business and Sci/Tech:**
- **Explanation:** The model struggles significantly when technology companies are involved in business activities (like lawsuits, product launches, or acquisitions). It tends to group tech-related keywords (like "Java", "DVD", "PCs", "Amazon", "Kodak") and misclassifies purely business oriented actions as Sci/Tech, or vice-versa when money/lawsuits are mentioned in a tech context.
- **Examples: **
  - *"Kodak Seeks \$1 Billion from Sun in Java Suit"* (True: Sci/Tech, Pred: Business/World)
  - *"Amazon launching DVD rentals"* (True: Business, Pred: Sci/Tech)
  - *"Lessons Airlines Can Learn From PCs"* (True: Business, Pred: Sci/Tech)

**2. Blurring between Business and World News:**
- **Explanation:** Macroeconomic indicators, government policies affecting markets, and national employment statistics blur the line between Business and World News. The model sometimes misinterprets global economic updates as general World news because they deal with countries and national statistics rather than specific corporate entities.
- **Examples:**
  - *"US economy generated 144,000 jobs in August (AFP)"* (True: Business, Pred: World)

**3. Keyword reliance / Lack of deep context:**
- **Explanation:** DistilBERT sometimes acts like a bag-of-words model, swayed by strong domain specific vocabulary even when the actual syntactic meaning of the sentence points to a different category. Metaphorical language or cross-domain topics confuse the classifier.
- **Examples:**
  - *"Ethics code written to reprogram tech industry"* (True: Business, Pred: Sci/Tech - confused by "reprogram" and "tech industry")